# Polarimetric signatures

- Open the San Francisco ALOS-1 scattering matrix.
- Convert `S` to `T3`.
- Apply 4 × 1 multilooking.
- Apply a 5 × 5 boxcar filter.
- Compute and plot polarimetric signatures for three example pixels.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar

from polsarpro.io import open_netcdf_beam
from polsarpro.polarisation import (
    plot_polarimetric_signature,
    polarimetric_signature,
)
from polsarpro.util import S_to_T3, boxcar, multilook, pauli_rgb

input_file = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")

In [ ]:
S = open_netcdf_beam(input_file)
S

## Prepare the coherency matrix

The row positions used below refer to the T3 matrix after 4 × 1 multilooking.

In [ ]:
T3 = S_to_T3(S)
T3 = multilook(T3, dim_az=4, dim_rg=1)
T3 = boxcar(T3, dim_az=5, dim_rg=5)
T3

## Compute four signatures

Each `(row, col)` pair is an integer pixel position in the filtered T3 matrix. The three selected points lie in areas with visibly different responses in the Pauli RGB image.

In [ ]:
points = {
    "Point A": (2100, 100),
    "Point B": (2340, 700),
    "Point C": (3900, 300),
}

signatures = {}
with ProgressBar():
    for name, (row, col) in points.items():
        signatures[name] = polarimetric_signature(T3, row=row, col=col)

## Inspect the selected pixels

The Pauli RGB image shows where the three example pixels lie in the filtered T3 matrix.

In [ ]:
with ProgressBar():
    pauli = pauli_rgb(T3).compute()

figure, axis = plt.subplots(figsize=(5, 7))
axis.imshow(pauli.transpose("y", "x", "band").values, origin="upper", aspect="auto")
for name, (row, col) in points.items():
    axis.scatter(col, row, s=45, edgecolor="white", label=name)
axis.set(title="Selected signature pixels", xlabel="Column", ylabel="Row")
axis.legend()
plt.show()

## Plot the signatures

`azimuth_angle` and `elevation_angle` control only the 3D camera. They are independent of the SAR image coordinates. The returned figure and axes can be edited with Matplotlib or saved with `figure.savefig(...)`.

In [ ]:
figures = {}
for name, signature in signatures.items():
    figure, axes = plot_polarimetric_signature(
        signature, azimuth_angle=-60, elevation_angle=30
    )
    figure.suptitle(name)
    figures[name] = (figure, axes)
plt.show()

For publication output, save any returned figure as PNG or PDF:

```python
figure, axes = figures["Point B"]
figure.savefig("point-b-signature.png", dpi=300, bbox_inches="tight")
figure.savefig("point-b-signature.pdf", bbox_inches="tight")
```